## Introduction
In the [universal decision-making framework](https://diogenesanalytics.com/blog/2026/03/05/universal-decision-making-framework) developed previously, action is evaluated in terms of its effect on position:

$$
U(P) = A(P) - T(P) + D^+(P) - D^-(P)
$$

In that formulation, position $P$ is the central object of decision-making. It represents the current state of the agent, while the utility function $U(P)$ evaluates the desirability of that state.

Before a position can be evaluated, however, a more fundamental question must be answered:

> **What information actually constitutes a position?**

This question is deceptively difficult because the world contains vastly more information than is relevant to decision-making.

Let

$$
\Omega
$$

denote the complete state of reality.

The world contains every fact about the current situation: physical properties, historical events, measurements, relationships, and countless other details. Only a small fraction of this information, however, is relevant to determining whether the agent is winning or losing.

We therefore distinguish between the complete world state and the information that is actually relevant for decision-making.

Let

$$
\omega = r(\Omega)
$$

denote the complete **decision-relevant information** extracted from the world.

The object $\omega$ represents the ideal position. It contains all information required to correctly evaluate the current state and determine the consequences of possible actions.

An actual agent, however, never has direct access to $\omega$. Instead, it operates using an approximation:

$$
P
$$

constructed from whatever information is currently available.

Decision-making therefore becomes a continual process of refining both the agent's representation of the world ($P$) and its method of evaluating that representation ($U$):

$$
P \rightarrow \omega,
\qquad
U \rightarrow U^*
$$

where $U^*$ represents the ideal evaluation function that would correctly evaluate the complete decision-relevant state.

If the ideal evaluation function were known, then improving the representation of position alone would cause the evaluation to converge:

$$
\lim_{P \to \omega} U^*(P) = U^*(\omega)
$$

This expression represents the pure **representation problem**: given a perfect evaluator, how accurately can the agent approximate the true position?

In reality, however, agents generally possess neither complete information nor a perfect evaluation function. Their decision process involves simultaneously improving both:

$$
\lim_{\substack{
P \to \omega \\
U \to U^*}
}
\lvert U(P)-U^*(\omega) \rvert = 0
$$

This expression represents the broader learning problem: accurate decision-making requires both an increasingly complete representation of the relevant state *and* an increasingly accurate method for evaluating that state.

This article concerns the first half of this problem.

It does **not** ask how utility should be computed. Instead, it asks a prior question:

> **What information must a position contain before it can be meaningfully evaluated?**

A common approach is to represent position as a collection of measurable variables,

$$
P = (x_1, x_2, \dots, x_n)
$$

but these variables are not the position itself. They are merely one possible **encoding** of the underlying information.

Let

$$
e(P)
$$

denote an arbitrary encoding of position.

Different encodings may employ different variables, units, or coordinate systems while preserving exactly the same decision-relevant information. The essential question is therefore not whether two representations use the same variables, but whether they preserve the same meaningful distinctions about the state.

Consequently, the central problem is not:

> Which variables should be included?

but:

> **Which information is fundamentally necessary, and which information is merely a redundant representation?**

This distinction separates two independent problems.

The first is the **representation problem**:

> What information constitutes position?

The second is the **evaluation problem**:

> How should utility be assigned to that position?

The present article addresses only the first problem. The second will be considered in future work.

To develop a principled criterion, we require a domain in which the relevant information can be identified exactly. The game of [Wordle](https://en.wikipedia.org/wiki/Wordle) provides such an environment. Because every state, action, and update rule is explicitly specified, competing representations of position can be tested directly rather than justified by intuition alone.

The goal of this article is to derive a general criterion for identifying the fundamental dimensions of position. The central claim is that position is not defined by the measurements we choose to record, but by the decision-relevant information that an agent must preserve in order to distinguish meaningful states of the world.

## The Problem of Identifying Position
If position is the decision-relevant information required to evaluate a state, then the central challenge is determining which information belongs in that representation.

In simple systems, this problem may appear trivial. A chess board, for example, can be represented by the location of every piece. A physical system can be represented by the positions and velocities of its components. A Wordle game can be represented by the remaining possible words.

However, real-world decision-making rarely provides such a clean representation.

An agent typically encounters a large collection of measurements, observations, and derived quantities, each of which may appear relevant.

For example, a person's financial position might be represented using:

$$
P =
(
\text{income},
\text{savings},
\text{assets},
\text{debt},
\text{credit score},
\text{net worth},
\dots
)
$$

A health position might include:

$$
P =
(
\text{weight},
\text{blood pressure},
\text{fitness},
\text{diet},
\text{sleep},
\dots
)
$$

A strategic position might include:

$$
P =
(
\text{resources},
\text{opportunities},
\text{constraints},
\text{relationships},
\dots
)
$$

Each of these variables may contain useful information. However, usefulness alone does not determine whether a quantity is a fundamental component of position.

A variable may be:

1. **fundamental information** required to distinguish meaningful states,

2. **a derived summary** computed from other information,

3. **a correlated indicator** that predicts outcomes without determining the state,

4. **a redundant representation** of information already contained elsewhere.

For example, net worth may be a useful financial metric, but it may simply summarize assets and liabilities. A credit score may predict financial outcomes, but it is itself derived from historical information. Entropy may measure uncertainty in a system, but it may not contain enough information to reconstruct the underlying state.

The difficulty is therefore not generating possible variables.

The difficulty is determining which information is irreducible.

Given a proposed representation:

$$
P = (x_1, x_2, \dots, x_n)
$$

we need a principled way to determine whether each component contains information that is genuinely required, or whether it can be removed without changing the agent's understanding of the state.

This is the fundamental representation problem:

> **How can we determine whether a piece of information belongs to position, rather than merely describing position?**

Answering this question requires a system where the complete decision-relevant state is known and candidate representations can be tested directly.

The game of [Wordle](https://en.wikipedia.org/wiki/Wordle) provides such an environment.

In [ ]:
from IPython.display import HTML
from collections import Counter
from typing import Literal

ColorCode = Literal["G", "Y", "B"]

COLOR: dict[ColorCode, str] = {
    "G": "#6aaa64",  # green (correct position)
    "Y": "#c9b458",  # yellow (correct letter, wrong position)
    "B": "#787c7e"   # gray (not in word)
}


def render_wordle(target: str, guess: str) -> HTML:
    """
    Render a Wordle-style feedback visualization for a single guess.

    This function implements the Wordle feedback operator:

        g(w*, a) → {🟩, 🟨, ⬛}^5

    where:
        - 🟩 = correct letter in correct position
        - 🟨 = correct letter in wrong position
        - ⬛ = letter not present in target

    Parameters
    ----------
    target : str
        The hidden target word (w*). Must be length 5.
    guess : str
        The guessed word (a_t). Must be length 5.

    Returns
    -------
    HTML
        A Jupyter-renderable HTML object showing the colored Wordle tiles.

    Notes
    -----
    The function uses a two-pass counting algorithm:
        1. Mark exact matches (greens) and decrement counts
        2. Assign partial matches (yellows) using remaining letter counts

    This ensures correct handling of repeated letters.
    """

    target = target.upper()
    guess = guess.upper()

    if len(target) != 5 or len(guess) != 5:
        raise ValueError("Both target and guess must be 5-letter words.")

    result: list[ColorCode] = ["B", "B", "B", "B", "B"]
    counts = Counter(target)

    # First pass: exact matches
    for i in range(5):
        if guess[i] == target[i]:
            result[i] = "G"
            counts[guess[i]] -= 1

    # Second pass: misplaced letters
    for i in range(5):
        if result[i] == "B" and counts[guess[i]] > 0:
            result[i] = "Y"
            counts[guess[i]] -= 1

    html = "<div style='display:flex;gap:4px;font-family:monospace;'>"

    for i in range(5):
        html += f"""
        <div style="
            width:40px;
            height:40px;
            display:flex;
            align-items:center;
            justify-content:center;
            background:{COLOR[result[i]]};
            color:white;
            font-weight:bold;
            font-size:18px;
        ">
            {guess[i]}
        </div>
        """

    html += "</div>"
    return HTML(html)

## Wordle as a Fully Specified Decision System
To determine what information is necessary for a position, we require a system where the evolution of information can be described exactly. Wordle provides such an environment.

At the beginning of a game of Wordle, the agent knows only that the hidden solution is a valid five-letter word.

Let:

$$
\mathcal C
$$

denote the finite set of **all valid candidate words**.

The hidden solution is:

$$
c^* \in \mathcal C
$$

At the beginning of the game, the agent has no information that distinguishes one candidate from another. Every word is therefore considered possible:

$$
S_0 = \mathcal C
$$

As the game progresses, guesses eliminate candidates that are inconsistent with the observed feedback. At time $t$, the remaining hypothesis space is:

$$
S_t \subseteq \mathcal C
$$

where each element of $S_t$ represents a candidate word that remains consistent with every observation obtained so far.

Since every possible information state is simply a subset of the candidate dictionary, the collection of all possible information states is the power set of $\mathcal C$:

$$
\mathcal S = 2^{\mathcal C}
$$

Each element of $\mathcal S$ therefore represents one possible **information state** of the agent—that is, one possible collection of candidate words consistent with the observations obtained so far.

To make this concrete, suppose:

$$
\mathcal C =
{
\text{CRANE},
\text{STAIN},
\text{PLANE}
}
$$

Then
$$
2^{\mathcal C} =
\left\{
\begin{aligned}
&
\emptyset,\\
&
\{\text{CRANE}\},\\
&
\{\text{STAIN}\},\\
&
\{\text{PLANE}\},\\
&
\{\text{CRANE},\text{STAIN}\},\\
&
\{\text{CRANE},\text{PLANE}\},\\
&
\{\text{STAIN},\text{PLANE}\},\\
&
\{\text{CRANE},\text{STAIN},\text{PLANE}\}
\end{aligned}
\right\}
$$

The current information state

$$
S_t
$$

is therefore one particular element of the state space

$$
\mathcal S
$$

Notice that the hidden solution itself never changes.

Throughout the game,

$$
c^*
$$

remains fixed.

What changes is the agent's information about that hidden solution. Each observation reduces uncertainty by eliminating candidate words that are inconsistent with the evidence accumulated so far.

The agent interacts with the system by selecting actions.

A guess is an action:

$$
a_t \in\mathcal C
$$

After playing $a_t$, the environment returns a feedback pattern comparing the guess with the hidden solution.

This feedback is generated by the observation function

$$
y_t = g(c^*, a_t)
$$

where:

$$
g:\mathcal C\times\mathcal C
\rightarrow
\{\text{🟩},\text{🟨},\text{⬛}\}^{5}
$$


maps a hidden word and a guess to the familiar five-tile Wordle response and:

* 🟩 indicates correct letter in correct position
* 🟨 indicates correct letter in incorrect position
* ⬛ indicates letter not present in the target word

Unlike a scalar reward or score, this observation is structured information. Each tile indicates whether the corresponding letter is correctly positioned, present elsewhere in the word, or absent from the hidden solution.

For example, if

$$
c^* = \text{CRANE}
$$

then different guesses produce different observations:

$a_t = \text{CRANE}$

In [ ]:
render_wordle("CRANE", "CRANE")

$a_t = \text{STAIN}$

In [ ]:
render_wordle("CRANE", "STAIN")

$a_t = \text{PLANE}$

In [ ]:
render_wordle("CRANE", "PLANE")

This feedback pattern is the observation:

$$
y_t=g(c^*,a_t)
$$

The observation is not the hidden word itself. Instead, it provides information that separates possible hidden worlds into those that are consistent with the observation and those that are not.

The feedback pattern is therefore a representation of information, not the information itself.

To understand what information the observation contains, we ask:

> **Which possible hidden words would have produced this same information if they were the true solution?**

Again consider the actual hidden word:

$$
c^*=\text{CRANE}
$$

and the guess:

$$
a_t=\text{STAIN}
$$

As we have seen this produces the feedback pattern:

In [ ]:
render_wordle("CRANE", "STAIN")

This is the observed feedback:

$$
y_t = g(\text{CRANE},\text{STAIN})
$$

At this point, the agent does not know that the hidden word is CRANE. It only knows the information encoded by this observation.

Now consider another possible word as the *potential hidden word*:

$$
c^* = \text{PLANE}
$$

If PLANE were the hidden solution, the same guess ($a_t = \text{STAIN}$) would produce:

In [ ]:
render_wordle("PLANE", "STAIN")

The resulting observation contains the same relevant constraints:

* the same letters are known to exist 
* the same letters are known to be absent
* the same positional restrictions are preserved

Therefore, from the perspective of the agent, CRANE and PLANE remain indistinguishable after this observation.

Both hidden worlds are consistent with the information revealed by the feedback.

Now consider:

$$
c^* = \text{BRICK}
$$

which with the same guess ($a_t = \text{STAIN}$) produces:

In [ ]:
render_wordle("BRICK", "STAIN")

This feedback contains different information.

It implies a different set of possible hidden words:

* different letters are present
* different letters are absent
* different constraints are imposed

Therefore, BRICK belongs to a different information class and can be eliminated.

The important distinction is that candidates are **not retained because their raw feedback patterns are visually identical**. They are retained because the feedback patterns contain equivalent information about the hidden state.

To formalize this, define an information extraction function:

$$
I(y)
$$

which maps an observation into the information contained within that observation.

For two observations:

$$
y_1, y_2
$$

we define an information equivalence relation:

$$
E(y_1,y_2) =
\begin{cases}
1,
&
I(y_1)=I(y_2)
\\
0,
&
\text{otherwise}
\end{cases}
$$

Thus, two observations are equivalent only when they preserve the same information about the hidden state.

The equality is therefore not:

$$
y_1 = y_2
$$

because two representations may differ while preserving the same information.

Instead, the relevant condition is:

$$
I(y_1) = I(y_2)
$$

or equivalently:

$$
E(y_1, y_2) = 1
$$

Returning to Wordle, suppose the observed feedback is:

$$
y_t = g(c^*, a_t)
$$

Now consider an arbitrary candidate:

$$
c \in S_t
$$

If $c$ were actually the hidden solution, the same guess would produce the hypothetical observation:

$$
g(c, a_t)
$$

The candidate survives exactly when this hypothetical observation contains the same information as the actual observation:

$$
E(g(c, a_t), g(c^*, a_t)) = 1
$$

Candidates failing this information-equivalence condition are removed from the hypothesis space.

Consequently, the transition function that allows us to update $S_t$ to $S_{t+1}$ is defined as:

$$
S_{t+1} = F(S_t, a_t)
$$

where:

$$
F(S_t, a_t) =
\left\{
c\in S_t
\;\middle|\;
E(g(c,a_t),g(c^*,a_t))=1
\right\}
$$

The transition therefore consists of filtering the current hypothesis space according to the information revealed by the latest observation.

Repeated application of this update rule generates a monotone refinement process:

$$
S_0
\supseteq
S_1
\supseteq
S_2
\supseteq
\cdots
\supseteq
S_T,
\qquad
\lvert S_T \rvert = 1
$$

Equivalently, the evolution of the system may be viewed as a deterministic process over the state space:

$$
\mathcal S = 2^{\mathcal C}
$$

where each action uniquely determines the next position:

$$
S_t \xrightarrow{a_t} S_{t+1}
$$

The mathematical structure of the Wordle environment is now completely specified. Every world state, observation, information transformation, and state transition can be computed exactly.

This makes Wordle an ideal laboratory for studying the representation problem introduced in the previous section. Rather than relying on intuition, we can directly test competing hypotheses about what information must be preserved in order to represent position.

The remaining question is therefore no longer how the game evolves, but how its information should be represented.

In the next section, we use this fully specified model to evaluate competing candidate representations of position and determine which information is fundamental, and which is merely a redundant description of the same underlying state.

## What Wordle Reveals About Position
The significance of the Wordle construction is not that it introduces another notion of position, but that it provides the first system in this article in which the transition function is known exactly.

Throughout the preceding sections, position was defined abstractly by the requirement that it preserve enough information to determine the transition

$$
P' = F(P,a)
$$

This definition immediately suggests a methodology.

Whenever we propose a representation of position, we are making a hypothesis about what information is fundamentally required to determine the evolution of the system.

That hypothesis can be tested.

Suppose we propose some representation $P_t$. To evaluate whether it is a valid representation of position, we ask a single question:

> **Given only $P_t$ and an action $a_t$, can we determine the next state $P_{t+1}$ under the transition function $F$?**

Formally, the representation is sufficient only if:

$$
P_{t+1}=F(P_t,a_t)
$$

is uniquely determined for every possible action $a_t$.

If the answer is **yes**, then the proposed representation preserves the transition dynamics of the system. If the answer is **no**, then it has discarded information required to determine future evolution and therefore cannot represent position.

Because the Wordle transition function is fully specified, we can perform this test directly on a variety of candidate representations.

| Proposed $P_t$                   | Can determine $F(P_t,a_t)$? | Valid? |
| :------------------------------: | :-------------------------: | :----: |
| Cardinality: $\lvert S_t \rvert$ |              ❌             |   ❌   | 
| Entropy: $H(S_t)$                |              ❌             |   ❌   |
| Letter-frequency statistics      |              ❌             |   ❌   |
| Sequence of guesses played       |              ❌             |   ❌   |
| Candidate set: $S_t$             |              ✅             |   ✅   |

Consider first cardinality of the candidate set (the number of remaining candidates) as position:

$$
P_t = \lvert S_t \rvert
$$

The number of remaining candidates appears to be a natural measure of position. It captures how much uncertainty remains and decreases as information is gained.

However, it does not contain enough information to determine the next state.

For example, two different candidate sets may both contain ten remaining words:

$$
\lvert S_t^{(1)} \rvert = \lvert S_t^{(2)} \rvert = 10
$$

Yet a guess $a_t$ may divide these two sets in completely different ways, producing different successor states:

$$
F(S_t^{(1)}, a_t) \neq F(S_t^{(2)}, a_t)
$$

Therefore the scalar quantity $\lvert S_t \rvert$ cannot determine the transition function. It describes the size of the state but not the structure that determines how the state evolves.

The same issue appears with entropy:

$$
P_t = H(S_t)
$$

Entropy provides a measure of uncertainty, but it is still only a summary statistic. Two candidate sets can have identical entropy while having different words, different letter structures, and therefore different responses to future guesses.

The representation captures a property of the state, but not the state information required to update it.

Letter-frequency statistics fail for the same reason. They preserve some information about the distribution of possible words:

$$
P_t = (f_A, f_B, \dots, f_Z)
$$

but they discard the relationships between letters and positions that determine Wordle feedback. Two candidate sets can have identical letter frequencies while producing different feedback patterns under the same guess.

The sequence of guesses presents a different failure mode:

$$
P_t = (a_1, a_2, \dots, a_t)
$$

The history of actions does contain information, but it does not directly specify the current state. The same sequence of guesses can lead to different candidate sets if the observed feedback differs, while different sequences of guesses can converge to the same candidate set.

The history is part of the path through the state space, not the state itself.

Finally, consider:

$$
P_t = S_t
$$

This representation succeeds because it preserves exactly the information needed by the transition function. Given the remaining candidate set and a new guess, the next state is uniquely determined:

$$
S_{t+1} = F(S_t, a_t)
$$

No additional information is required, and no relevant information has been discarded.

The important observation is that these representations are not evaluated according to usefulness, predictive power, or convenience. Several failed representations are extremely useful in practice. The number of remaining candidates measures progress. Entropy can guide exploration. Letter frequencies can help select guesses.

However, usefulness is not the criterion.

The criterion is whether the representation preserves the transition structure of the system.

Wordle therefore provides more than an example of position. It provides a controlled environment in which competing hypotheses about the *representation of position* can be tested against an explicitly known transition function:

$$
S_{t+1} = F(S_t, a_t) = \{ w \in S_t \mid g(w, a_t) = g(w^*, a_t) \}
$$

This transforms the question of position from one of intuition into one of verification. Rather than asking whether a variable seems important, we ask whether the information it contains is necessary to reproduce the evolution of the system.

The lesson extends beyond Wordle. In more complex domains, the transition function may not be explicitly known, making this test more difficult to perform. Nevertheless, the principle remains unchanged: a proposed representation of position is valid only if it preserves the information required to determine the system's future evolution under every possible action.

## A Criterion for Fundamental Dimensions of Position
The previous section established a general methodology for evaluating proposed representations of position. Rather than asking whether a representation appears useful or intuitive, we ask a more fundamental question:

> **Does it preserve the transition function?**

The Wordle example demonstrated how this question can be answered when the transition function is known exactly. More importantly, it suggests a criterion that is independent of Wordle itself.

Suppose a system is described by a transition function

$$
F : \mathcal P \times \mathcal A \rightarrow \mathcal P
$$

Now consider two representations of the same underlying state,

$$
P
\quad\text{and}\quad
P'
$$

If every action produces exactly the same successor position,

$$
F(P,a)=F(P',a),
\qquad
\forall a\in\mathcal A
$$

then the two representations are indistinguishable from the perspective of the system's dynamics. Although they may differ in how the state is encoded, they preserve exactly the same evolution under every possible action.

This defines an equivalence relation over representations:

$$
P
\sim
P'
\quad\Longleftrightarrow\quad
F(P,a)=F(P',a)
\quad
\forall a\in\mathcal A
$$

Position should therefore not be identified with any particular coordinate system. Rather, a position is the equivalence class of all representations that induce the same transition dynamics.

This immediately yields a criterion for identifying fundamental dimensions.

Suppose

$$
P=(x_1,x_2,\ldots,x_n)
$$

Consider removing one component to obtain

$$
P_{\setminus x_i}
$$

If

$$
F(P,a)=
F(P_{\setminus x_i},a)
\qquad
\forall a\in\mathcal A
$$

then removing $x_i$ has no effect on the evolution of the system. The information contained in $x_i$ is already present elsewhere in the representation, making it redundant.

Conversely, if there exists at least one action for which

$$
F(P,a)
\neq
F(P_{\setminus x_i},a)
$$

then the removal of $x_i$ changes the transition dynamics. In that case, $x_i$ contains information that cannot be recovered from the remaining components and is therefore a fundamental dimension of position.

We therefore obtain the central result of this article.

> **A component of position is fundamental if and only if removing it changes the transition function for at least one possible action.**

Equivalently,

$$
x_i
\text{ is fundamental}
\quad\Longleftrightarrow\quad
\exists a\in\mathcal A
\text{ such that }
F(P,a)
\neq
F(P_{\setminus x_i},a)
$$

This criterion is independent of any particular application. It does not depend on how useful a variable appears, how predictive it is, or how easily it can be measured. It depends only on whether the information contained in that variable is necessary to preserve the action-dependent evolution of the system.

## Conclusion

This article has focused on the representation problem: determining what information constitutes a position. The central result is that position is not defined by the variables we happen to measure, but by the minimal information required to preserve the transition structure of the system. A valid representation of position must contain enough information that, given any action, the resulting successor state is fully determined.

The Wordle example demonstrates this principle in a fully specified environment. The candidate set $S_t$ is not merely one convenient representation among many; it is the structure required to determine the future evolution of the game. Simpler quantities such as candidate count, entropy, or letter frequencies may provide useful summaries, but they do not preserve the complete transition dynamics.

However, identifying position is only one part of decision-making. Once the position has been determined, a separate question remains: how should different positions be evaluated?

In the [general framework](https://diogenesanalytics.com/blog/2026/03/05/universal-decision-making-framework), this question is represented by the utility function:

$$
U(P)=A(P)-T(P)+D^+(P)-D^-(P)
$$

The present article establishes what the argument $P$ must contain before such a function can be meaningfully applied. A future extension of this framework must address the complementary problem: how utility functions are constructed over positions.

The distinction between these two problems is fundamental. Position describes the information state of the agent: what is currently known and what future states remain reachable. Utility evaluates that position according to the objectives, preferences, or goals of the agent. A complete decision framework therefore requires both a representation of the state space and a method for assigning value within that space.

Wordle provides a simple illustration of this separation. Once position is represented by the candidate set $S_t$, additional functions can be defined over that structure to measure properties of the position. For example, an information-based progress metric could measure the reduction of uncertainty:

$$
I(S_t)=\log\frac{\lvert\mathcal{W}\rvert}{\lvert S_t\rvert}
$$

This quantity rewards positions in which the remaining hypothesis space has been reduced. It is not a complete theory of utility, but it demonstrates a more general principle: once the informational structure of a position has been correctly identified, it becomes possible to define functions that evaluate the quality of the states an agent reaches.

The deeper challenge is therefore not only identifying what a position is, but determining how an agent should value the positions it can reach.